# PT-Flow — experimental resultsProduces every figure and table in one pass, into `/kaggle/working/results/`.| # | Result | Output | Claim it supports ||---|---|---|---|| **A1** | Closed-form estimator checks | table | the estimator is *correct* — φ₀, prox and S⁻¹ = (1+τ)I match closed forms || **A4** | R0 variance reversal | **figure** | Theorem 2.9: naive spread diverges as ε cools, tilted vanishes on the √ε law || **B** | λ_prox ablation in 2-D | table + figure | the PT term does not degrade Mode A, the deployed sampler || **C1** | Loss decomposition at d = 3072 | table | the PT term reaches 150–8000× the OT anchor || **C2** | Guard degeneracy, d = 2 vs d = 3072 | table | the log-weight clip pins ESS, so the health monitor certifies unconditionally || **C3** | Two-arm resume ablation | **figure** | the corrected configuration restores an honest certificate || **D1** | Matched-seed sample grids | **figure** | what 1 → 1000 → 25000 steps actually produced |A1, A4 and B are CPU-only and take about fifteen minutes together. C1 and C2 readthe run you already have. C3 is the only cell that trains; it is gated on`RUN_C3` and costs roughly 3 h of GPU.**The claim this set supports** — worth stating in these terms rather than"the method works":> The implementation is verified correct against closed forms (A1) and reproduces> the predicted variance reversal (A4). In 2-D, where the method's published> validation sits, the PT term does not degrade the deployed sampler (B). At> d = 3072 the estimator's guards degenerate: the log-weight clip pins ESS to a> constant, so the health monitor certifies unconditionally (C1, C2). The> corrected configuration restores an honest certificate (C3).

## [0] Knobs and paths

In [ ]:
import os, globREPO_GDRIVE_ID = "1S8SsNAE6AqQ_QHSTYQJTuZ6gdNhyAixR"CIFAR_GDRIVE_ID = "1AMESa9etn7VmnXb3GPbO2DWYttULbRvY"WORK     = "/kaggle/working/ptflow"DATA     = "/kaggle/working/data/cifar10"DOWNLOAD = "/kaggle/working/_downloads"RESULTS  = "/kaggle/working/results"FIGS     = os.path.join(RESULTS, "figures")TABS     = os.path.join(RESULTS, "tables")# --- C3: the only cell that trains -----------------------------------------RUN_C3     = True     # False -> skip; everything else still runs (~15 min, CPU)STEPS_C3   = 1500     # per arm, on top of the 25000 already done. ~1.6 h each.RESUME_STEP = 25000# --- model knobs: these MUST match the run that produced the checkpoints ----# (the defaults of ptflow_cifar10_kaggle_t4x2.ipynb)HIDDEN, DEPTH, HEADS, PATCH = 384, 8, 6, 2NUM_CLASSES = 10CIFAR_NAMES = ["airplane", "automobile", "bird", "cat", "deer",               "dog", "frog", "horse", "ship", "truck"]for d in (DOWNLOAD, RESULTS, FIGS, TABS, os.path.dirname(DATA)):    os.makedirs(d, exist_ok=True)# --- locate the attached checkpoints and metrics ----------------------------CKPTS = {}for p in sorted(glob.glob("/kaggle/input/**/state_*.pt", recursive=True)):    try:        CKPTS[int(os.path.basename(p).split("_")[-1].split(".")[0])] = p    except ValueError:        passMETRICS = (sorted(glob.glob("/kaggle/input/**/metrics*.json*", recursive=True))           + sorted(glob.glob("/kaggle/working/**/metrics*.json*", recursive=True)))print("checkpoints found:")for s, p in sorted(CKPTS.items()):    print(f"  step {s:>6}  {p}")if not CKPTS:    print("  NONE -- attach the checkpoint dataset (D1 and C3 will be skipped)")print("\nmetrics found:", METRICS[0] if METRICS else "NONE -- C1/C2 will be skipped")print("       (add metrics.json to the same Kaggle dataset if missing)")

## [1] Environment and repo

In [ ]:
import subprocess, sys, os, shutil, zipfile, tarfile, glob, torchsubprocess.run([sys.executable, "-m", "pip", "install", "-q",                "einops", "absl-py", "gdown"], check=True)def gdrive_download(file_id, dest):    if os.path.exists(dest) and os.path.getsize(dest) > 1024:        print("cached:", dest); return dest    import gdown    gdown.download(id=file_id, output=dest, quiet=False)    if not os.path.exists(dest):        raise SystemExit("Drive download failed -- Kaggle needs Internet ON.")    return destdef unpack(path, dest, depth=0):    os.makedirs(dest, exist_ok=True)    with open(path, "rb") as f:        magic = f.read(4)    if magic[:2] == b"PK":        with zipfile.ZipFile(path) as z: z.extractall(dest)    elif magic[:2] == b"\x1f\x8b" or tarfile.is_tarfile(path):        with tarfile.open(path) as t:            try: t.extractall(dest, filter="data")            except TypeError: t.extractall(dest)    else:        raise ValueError(f"{path}: neither zip nor tar")    if depth == 0:        for f in (glob.glob(os.path.join(dest, "**", "*.tar.gz"), recursive=True)                  + glob.glob(os.path.join(dest, "**", "*.tgz"), recursive=True)):            unpack(f, os.path.dirname(f), depth=1)    return destdef find_repo_root(root):    for dp, dn, fn in os.walk(root):        if "train.py" in fn and "pt" in dn and "models" in dn:            return dp    raise SystemExit(f"no PT-Flow checkout under {root}")if not os.path.isdir(os.path.join(WORK, "pt")):    z = gdrive_download(REPO_GDRIVE_ID, os.path.join(DOWNLOAD, "repo.zip"))    ex = os.path.join(DOWNLOAD, "repo_extract")    shutil.rmtree(ex, ignore_errors=True)    unpack(z, ex)    shutil.copytree(find_repo_root(ex), WORK)os.chdir(WORK)if WORK not in sys.path:    sys.path.insert(0, WORK)os.environ["DRIFT_COMPILE"] = "0"# utils/env.py ships /path/to placeholders; point them somewhere writable.import pathlib, textwrappathlib.Path(WORK, "utils", "env.py").write_text(textwrap.dedent(f"""\    from __future__ import annotations    HF_REPO_ID = "Goodeat/drifting"    HF_ROOT = "/kaggle/working/_hf/drifting"    VAE_HF_PATH = "/kaggle/working/_hf/sdvae"    TORCH_HUB_DIR = "/kaggle/working/_hf/torch_hub"    BASELINE_HF_REPO_ID = "jiaqihan99/W-Flow"    BASELINE_HF_ROOT = "/kaggle/working/_hf/baseline"    IMAGENET_PATH = {str(DATA)!r}    IMAGENET_CACHE_PATH = "/kaggle/working/_hf/latent_cache"    IMAGENET_FID_NPZ = "/kaggle/working/_hf/fid_stats.npz"    IMAGENET_PR_NPZ = ""    """), encoding="utf-8")import matplotlib.pyplot as pltplt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200,                     "savefig.bbox": "tight", "axes.grid": True,                     "grid.alpha": 0.3, "font.size": 10})def save_table(name, header, rows):    """CSV to disk, markdown to stdout -- paste-ready for the report."""    import csv    p = os.path.join(TABS, f"{name}.csv")    with open(p, "w", newline="", encoding="utf-8") as f:        w = csv.writer(f); w.writerow(header); w.writerows(rows)    w = [max(len(str(header[i])), max((len(str(r[i])) for r in rows), default=0))         for i in range(len(header))]    print("| " + " | ".join(str(h).ljust(w[i]) for i, h in enumerate(header)) + " |")    print("|" + "|".join("-" * (x + 2) for x in w) + "|")    for r in rows:        print("| " + " | ".join(str(c).ljust(w[i]) for i, c in enumerate(r)) + " |")    print(f"\n-> {p}")    return pprint("\ntorch", torch.__version__, "| cuda", torch.cuda.is_available(),      "|", torch.cuda.device_count(), "gpu(s)")

## [A1] Closed-form estimator checksThe quadratic case has every object in the paper available exactly — ψ₀, φ₀, theprox, and S⁻¹ = (1+τ)I. If these pass, the estimator is right on this build ofPyTorch. No GPU, no data, no downloaded weights.

In [ ]:
import subprocess, sys, os, reenv = dict(os.environ); env["PYTHONPATH"] = WORK; env["DRIFT_COMPILE"] = "0"r = subprocess.run([sys.executable, os.path.join("tests", "test_math.py")],                   cwd=WORK, env=env, capture_output=True, text=True)out = r.stdoutprint(out[-3000:])with open(os.path.join(TABS, "A1_test_math_log.txt"), "w", encoding="utf-8") as f:    f.write(out + "\n" + r.stderr)# One row per numbered section.  tests/test_math.py prints "  [ok  ] name"# or "  [FAIL] name" per check, and "N. Section title" per group.sections, cur = [], Nonefor line in out.splitlines():    t = line.strip()    m = re.match(r"^(\d+)\.\s+(\D.*)$", t)    if m:        cur = [m.group(2).strip(), 0, 0]; sections.append(cur)    elif cur is not None and t.startswith("["):        cur[1 if t.startswith("[ok") else 2] += 1tot = re.search(r"(\d+)\s+passed,\s+(\d+)\s+failed", out)rows = [[s[0], s[1] + s[2], "PASS" if s[2] == 0 else f"{s[2]} FAILED"] for s in sections]if tot:    rows.append(["TOTAL", int(tot.group(1)) + int(tot.group(2)),                 "PASS" if tot.group(2) == "0" else f"{tot.group(2)} FAILED"])save_table("A1_closed_form_checks", ["check group", "n", "result"], rows)print("\nexit code:", r.returncode)

## [A4] R0 — the variance reversalTheorem 2.9's claim, and the reason the method is feasible at all:```naive   s² ~ |grad phi|² / 2eps     -> diverges as eps coolstilted  s² ~ (5/6) d M_3^2 eps      -> vanishes as eps cools, on a sqrt(eps) law```The dotted line is the predicted √ε slope, not a fit.

In [ ]:
import math, os, torch, numpy as np, matplotlib.pyplot as pltfrom ptflow.estimator import naive_phi0, tilted_phi0from ptflow.potential import phi_grad# tests/ has no __init__.py, so `from tests.test_math import ...` can lose# the sys.path scan to a regular `tests` package elsewhere in the image.  Load# the repo's own file directly instead -- same class, no copy, no new files.try:    from tests.test_math import MildlyCubicPotentialexcept ModuleNotFoundError:    import importlib.util    _spec = importlib.util.spec_from_file_location(        "_pt_math_ref", os.path.join(WORK, "tests", "test_math.py"))    _m = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_m)    MildlyCubicPotential = _m.MildlyCubicPotential    print("loaded MildlyCubicPotential by file path")torch.manual_seed(0)B, tau, kappa = 128, 0.5, 0.4pot = MildlyCubicPotential(tau, kappa)c  = torch.zeros(B, dtype=torch.long)x0 = torch.randn(B, 4, 4, 2) * 0.5epss = [0.4, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005]naive_s, tilt_s = [], []for eps in epss:    m0 = x0.clone()    for _ in range(40):                      # solve for the true prox        g, _ = phi_grad(pot, m0, c)        m0 = x0 - g    s0 = -torch.log((1.0 + tau + kappa * m0).clamp_min(1e-3))   # S^-1 = I + H    naive_s.append(naive_phi0(pot, x0, c, eps, K=64).logw_spread.mean().item())    tilt_s.append(tilted_phi0(pot, x0, c, m0, s0, eps, K=64,                              alpha_def=0.0, logw_clip=0.0).logw_spread.mean().item())    print(f"  eps={eps:<7g} naive={naive_s[-1]:9.3f}   tilted={tilt_s[-1]:.6f}")pred = [tilt_s[0] * math.sqrt(e / epss[0]) for e in epss]fig, ax = plt.subplots(figsize=(6.6, 4.6))ax.loglog(epss, naive_s, "o-", label=r"naive  $\mathcal{N}(x_0,\,2\varepsilon I)$")ax.loglog(epss, tilt_s, "s-", label="prox-tilted (Laplace-matched)")ax.loglog(epss, pred, "k:", label=r"predicted $\propto\sqrt{\varepsilon}$")ax.set_xlabel(r"$\varepsilon$"); ax.set_ylabel("log-weight spread  $s_{res}$")ax.set_title("R0: variance reversal (Theorem 2.9)")ax.invert_xaxis(); ax.legend(); fig.tight_layout()fig.savefig(os.path.join(FIGS, "A4_variance_reversal.png")); plt.show()ratio_obs  = tilt_s[0] / tilt_s[-1]ratio_pred = math.sqrt(epss[0] / epss[-1])save_table("A4_variance_reversal",           ["eps", "naive spread", "tilted spread", "predicted sqrt(eps)"],           [[f"{e:g}", f"{n:.4f}", f"{t:.6f}", f"{p:.6f}"]            for e, n, t, p in zip(epss, naive_s, tilt_s, pred)])print(f"\ntilted spread shrinks {ratio_obs:.2f}x over a {epss[0]/epss[-1]:.0f}x eps range; "      f"sqrt law predicts {ratio_pred:.2f}x")print(f"naive spread grows {naive_s[-1]/naive_s[0]:.1f}x over the same range -- "      "the reversal, in one line")# The tail of the sweep is the precision floor README_PTFLOW.md documents: the# sqrt law holds down to eps ~ 0.01 and then flattens, which is where the# estimator stops being variance-limited and starts being fp32-limited.tail = [(e, t) for e, t in zip(epss, tilt_s) if e <= 0.02]print("\nprecision floor (README records 0.180 -> 0.140 -> 0.123 here):")for (e1, t1), (e2, t2) in zip(tail, tail[1:]):    print(f"  eps {e1:g} -> {e2:g}:  {t1:.3f} -> {t2:.3f}   "          f"observed {t1/t2:.2f}x vs sqrt-law {math.sqrt(e1/e2):.2f}x")

## [B] λ_prox ablation in 2-DEight Gaussians, 3000 steps, one seed each. Energy distance to the data, lower isbetter. **λ_prox = 0 is the baseline** every other row is measured against.This is the regime the method's published validation sits in: `d = 2`, where thelog-weight spread stays O(0.1) and the estimator is genuinely healthy. Keep thatin mind when reading C2.

In [ ]:
import json, subprocess, sys, osLAMBDAS = [0.0, 0.3, 1.0, 3.0]rows = []for lam in LAMBDAS:    out = f"/kaggle/working/toy2d_lam{lam}"    cmd = [sys.executable, os.path.join("examples", "toy2d.py"),           "--steps", "3000", "--batch", "256", "--lambda-prox", str(lam),           "--device", "cpu", "--out", out]    if lam == 0.3:        cmd.append("--plot")    print(f"--- lambda_prox = {lam} " + "-" * 40)    r = subprocess.run(cmd, cwd=WORK, env=env, capture_output=True, text=True)    if r.returncode != 0:        print(r.stdout[-1500:]); print(r.stderr[-1500:]); continue    res = json.load(open(f"{out}/results.json"))    hist = json.load(open(f"{out}/history.json"))    rows.append([f"{lam:g}" + ("  (= the OT-drift baseline)" if lam == 0 else ""),                 f"{res['mode_A']:.5f}", f"{res['mode_B']:.5f}", f"{res['mode_C']:.5f}",                 f"{res['final_ess']:.3f}", f"{res['final_eps']:.3f}",                 f"{min(hist['logw_spread']):.3f}-{max(hist['logw_spread']):.3f}"                 if "logw_spread" in hist else "n/a"])    print(f"  A={res['mode_A']:.5f}  B={res['mode_B']:.5f}  C={res['mode_C']:.5f}  "          f"ESS={res['final_ess']:.3f}")save_table("B_lambda_prox_ablation_2d",           ["lambda_prox", "Mode A (1-NFE)", "Mode B (8-NFE)", "Mode C (SNIS K=32)",            "final ESS", "final eps", "logw_spread range"], rows)import shutilsrc = "/kaggle/working/toy2d_lam0.3/toy2d.png"if os.path.exists(src):    shutil.copy2(src, os.path.join(FIGS, "B_toy2d_lambda0.3.png"))    from IPython.display import Image, display    display(Image(filename=src))print("""How to read it:  * Mode A must not get WORSE as lambda_prox rises -- A is the deployed sampler.  * Mode B only helps once the prox identity m = prox_phi actually holds;    watch prox_resid_rel, not the energy distance.  * Mode C reweights rather than moves, so it cannot walk samples off the manifold.""")

## [C1] What is actually training the generator at d = 3072`λ_prox = 0.3` looks small, but `prox_loss` is unnormalised-large while the driftloss is O(0.5). The ratio column is the finding.

In [ ]:
import json, statistics as st, osrows_m = []if METRICS:    with open(METRICS[0]) as f:        rows_m = [json.loads(l) for l in f if l.strip()]    print(f"{len(rows_m)} logged rows, steps {rows_m[0]['step']} -> {rows_m[-1]['step']}")else:    print("no metrics file attached -- C1 and C2 skipped")if rows_m:    tab = []    for r in rows_m[::max(1, len(rows_m) // 8)]:        d  = r["loss_0.05/global"]        pt = r["pt/lambda_prox"] * r["pt/loss_prox"]        tab.append([r["step"], f"{d:.4f}", f"{pt:.1f}", f"{r['loss']:.1f}", f"{pt/d:.0f}x"])    save_table("C1_loss_decomposition",               ["step", "OT drift loss", "lambda*prox_loss", "total", "PT / drift"], tab)    dr = st.mean(r["loss_0.05/global"] for r in rows_m)    pt = st.mean(r["pt/lambda_prox"] * r["pt/loss_prox"] for r in rows_m)    print(f"\nmean over the run: drift {dr:.4f}   PT {pt:.1f}   ratio {pt/dr:.0f}x")    print(f"the baseline arm is {100*dr/(dr+pt):.3f}% of the generator's objective")

## [C2] Guard degeneracy: d = 2 versus d = 3072The clip is applied to `log_w` *before* ESS is measured([ptflow/estimator.py:282](../ptflow/ptflow/estimator.py)). With K = 8, `torch.median`returns the 4th of 8, so exactly 4 weights sit above it; they are flattened to acommon ceiling while the other 4 fall ≥30 below and weigh e⁻³⁰. That givesESS = (4c)²/(8·4c²) = 0.5 **for any spread above the clip**.

In [ ]:
import statistics as stif rows_m:    e = [r["pt/ess"] for r in rows_m]    cf = [r["pt/clip_frac"] for r in rows_m]    ident = sum(1 for a, b in zip(e, cf) if abs(a - b) < 1e-9)    toy_spread = f"{min(tilt_s):.3f}-{max(tilt_s):.3f}"    save_table("C2_guard_degeneracy",        ["quantity", "toy  d = 2", "CIFAR  d = 3072", "healthy per README"],        [["log-weight spread", toy_spread,          f"{st.mean(r['pt/logw_spread'] for r in rows_m):.3g}", "O(1), shrinking as eps cools"],         ["ESS / K", "0.375 - 0.547 (varies)",          f"{st.mean(e):.5f} (pinned)", "> 0.30"],         ["distinct ESS values", "continuous",          f"{len(set(round(x,6) for x in e))} in {len(e)} rows", "continuous"],         ["ESS == clip_frac", "n/a (clip inactive)",          f"{ident}/{len(e)} rows identical", "never"],         ["clip_frac", "0.000",          f"{st.mean(cf):.5f}", "0"],         ["prox_resid_rel", "~3 (README toy)",          f"{st.mean(r['pt/prox_resid_rel'] for r in rows_m):.2f}", "< 0.3"],         ["|grad phi| vs displacement", "comparable",          f"{st.mean(r['pt/prox_resid'] for r in rows_m):.0f} vs "          f"{st.mean(r['pt/displacement'] for r in rows_m):.0f}", "comparable"],         ["log_scale_mean", "adaptive",          f"{rows_m[-1]['pt/log_scale_mean']:.4f} "          f"({len(set(r['pt/log_scale_mean'] for r in rows_m))} distinct)", "adaptive"],         ["health state", "2 / 1 / 0 as ESS moves",          f"2 in {sum(1 for r in rows_m if r['pt/health']==2)}/{len(rows_m)} rows",          "responsive"]])    # the trace that makes the point visually    import matplotlib.pyplot as plt    step = [r["step"] for r in rows_m]    fig, ax = plt.subplots(1, 3, figsize=(15, 4))    ax[0].plot(step, e, lw=1.2, label="pt/ess")    ax[0].plot(step, cf, lw=1.2, ls="--", label="pt/clip_frac")    ax[0].axhline(0.30, color="tab:orange", ls=":", label="healthy > 0.30")    ax[0].set_title("ESS and clip_frac are the same number"); ax[0].legend(fontsize=8)    ax[1].semilogy(step, [r["pt/logw_spread"] for r in rows_m], lw=1.2)    ax[1].axhline(30, color="tab:red", ls="--", label="logw_clip = 30")    ax[1].set_title("log-weight spread vs the clip"); ax[1].legend(fontsize=8)    ax[2].semilogy(step, [r["pt/prox_resid_rel"] for r in rows_m], lw=1.2)    ax[2].axhline(0.3, color="tab:orange", ls="--", label="usable < 0.3")    ax[2].set_title("prox_resid_rel: is m = prox_phi?"); ax[2].legend(fontsize=8)    for a in ax: a.set_xlabel("step")    fig.suptitle("Guard degeneracy at d = 3072", fontsize=12)    fig.tight_layout()    fig.savefig(os.path.join(FIGS, "C2_guard_degeneracy.png")); plt.show()

## [C3] Two-arm resume ablationBoth arms branch from the **same** `state_00025000.pt`, so the comparison iscontrolled. `prox_progress` is already 1.0 in the checkpoint, so each arm's new`lambda_prox_max` takes effect immediately — no re-warmup.| arm | change | hypothesis ||---|---|---|| **A** | `lambda_prox_max: 0` | the pure baseline. The drift loss falls faster once the PT term stops dominating. || **B** | `lambda_prox_max: 1e-4`, `scale_mode: none`, `logw_clip: 0` | ESS drops off the pinned 0.5 toward 1/K and the monitor reports honestly. |All four changes are config keys. Nothing under `ptflow/` is modified.

In [ ]:
import os, sys, glob, json, shutil, yaml, pathlib, subprocess, timedef base_config(total_steps, lam, scale_mode="learned", logw_clip=30.0):    cfg = {      "logging": {"use_wandb": False, "log_every_k": 10},      "dataset": {"resolution": 32, "use_aug": False, "use_latent": False,                  "use_cache": False, "num_classes": NUM_CLASSES,                  "batch_size": 256, "eval_batch_size": 256,                  "kwargs": {"num_workers": 2, "pin_memory": True, "prefetch_factor": 2}},      "model": {"cond_dim": HIDDEN, "input_size": 32, "in_channels": 3,                "out_channels": 3, "patch_size": PATCH, "hidden_size": HIDDEN,                "depth": DEPTH, "num_heads": HEADS, "mlp_ratio": 4.0,                "use_qknorm": True, "use_swiglu": True, "use_rope": True,                "use_rmsnorm": True, "n_cls_tokens": 4, "noise_classes": 64,                "noise_coords": 32, "use_bf16": False, "attn_fp32": True,                "residual": True},      "optimizer": {"lr_schedule": {"learning_rate": 2.0e-4, "warmup_steps": 100,                                    "lr_schedule": "const", "total_steps": total_steps},                    "weight_decay": 0.01, "adam_b1": 0.9, "adam_b2": 0.95},      "train": {"diverse_noise": True, "train_batch_size": 16, "seed": 42,                "total_steps": total_steps, "save_per_step": max(500, total_steps),                "keep_every": total_steps, "keep_last": 2, "eval_per_step": 10**9,                "pos_per_sample": 16, "neg_per_sample": 8,                "positive_bank_size": 64, "negative_bank_size": 512,                "forward_dict": {"gen_per_label": 16, "cfg_min": 1.0,                                 "cfg_max": 4.0, "neg_cfg_pw": 3.0},                "activation_kwargs": {}, "loss_kwargs": {"R_list": [0.05]},                "cfg_list": [1.0], "ema_decay": 0.999, "push_per_step": 128,                "push_at_resume": 20, "grad_accum_steps": 2, "ot_mode": "debiased",                "ot_kwargs": {"sinkhorn_num_iter": 10, "use_new_cfg": True,                              "disable_diag_mask": True, "resample_neg": True,                              "batch_sinkhorn": False, "use_quadratic_cost": True}},      "feature": {"use_mae": False, "use_convnext": False, "convnext_bf16": False},      "pt": {"enabled": True, "K": 8, "noise_bsz": 8, "data_bsz": 16,             "p_uncond": 0.1, "potential_chunk": 0, "prox_max_batch": 32,             "prox_mode": "detach", "scale_K": 4, "lambda_gauge": 0.1,             "lambda_mag": 0.0, "logw_clip": logw_clip, "max_grad_norm": 1.0,             "curv_probe": False, "curv_allow": 0.5, "ema_decay": 0.999,             "init_generator_from": "",             "model": {"cond_dim": HIDDEN, "hidden_size": 256, "depth": 6,                       "num_heads": 4, "patch_size": PATCH, "mlp_ratio": 4.0,                       "use_qknorm": True, "use_swiglu": True, "use_rope": True,                       "use_rmsnorm": True, "n_cls_tokens": 0, "use_bf16": False,                       "attn_fp32": True, "phi_scale": None, "quad_anchor": 0.0,                       "output_mode": "phi"},             "optimizer": {"weight_decay": 0.01, "adam_b1": 0.9, "adam_b2": 0.95,                           "lr_schedule": {"learning_rate": 1.0e-4, "warmup_steps": 100,                                           "lr_schedule": "const", "total_steps": total_steps}},             "schedule": {"eps_max": 0.2, "eps_min": 0.005, "eps_anneal_steps": 600,                          "eps_warmup": 50, "alpha_def_start": 0.1,                          "alpha_def_end": 0.01, "alpha_def_degraded": 0.25,                          "prox_warmup": 150, "prox_ramp": 350,                          "lambda_prox_max": lam, "prox_decay_on_degrade": 0.999,                          "lambda_scale": 0.1, "lambda_curv": 0.0,                          "theta_period": 1, "theta_period_degraded": 2,                          "ess_healthy": 0.3, "ess_broken": 0.05, "ess_ema_decay": 0.98},             "scale_model": {"cond_dim": HIDDEN, "hidden_size": 128, "depth": 3,                             "num_heads": 4, "patch_size": PATCH, "mlp_ratio": 4.0,                             "use_qknorm": True, "use_swiglu": True, "use_rope": True,                             "use_rmsnorm": True, "n_cls_tokens": 0, "use_bf16": False,                             "attn_fp32": True, "scale_max": 3.0},             "scale_mode": scale_mode},    }    return cfgCFG_REF = os.path.join(WORK, "configs", "gen", "_ref.yaml")pathlib.Path(CFG_REF).write_text(yaml.safe_dump(base_config(RESUME_STEP, 0.3), sort_keys=False))# --- preflight: does the checkpoint match these model knobs? ----------------if RESUME_STEP in CKPTS:    import torch    from models.generator import DitGen    pay = torch.load(CKPTS[RESUME_STEP], map_location="cpu", weights_only=False)    try:        DitGen(num_classes=NUM_CLASSES, **base_config(1, 0.0)["model"]).load_state_dict(            pay["ema_model"], strict=True)        print(f"preflight OK: state_{RESUME_STEP:08d}.pt matches "              f"HIDDEN={HIDDEN} DEPTH={DEPTH} HEADS={HEADS} PATCH={PATCH}")    except RuntimeError as exc:        print("PREFLIGHT FAILED -- the model knobs in cell [0] do not match the checkpoint.")        print(str(exc)[:900])        blocks = {int(k.split(".")[2]) for k in pay["ema_model"] if k.startswith("model.blocks.")}        adaln = [v.shape for k, v in pay["ema_model"].items() if k.endswith("adaLN_mod.1.weight")]        print(f"\nhint: checkpoint has {len(blocks)} blocks; adaLN shapes {adaln[:1]} "              f"-> hidden_size = {adaln[0][1] if adaln else '?'}")    del pay

In [ ]:
import os, sys, time, shutil, glob, pathlib, yaml, subprocessARMS = {    "A_baseline_only":  dict(lam=0.0,    scale_mode="learned", logw_clip=30.0),    "B_corrected":   dict(lam=1e-4,   scale_mode="none",    logw_clip=0.0),}ARM_DIRS = {k: f"/kaggle/working/runs/arm_{k}" for k in ARMS}if not (RUN_C3 and RESUME_STEP in CKPTS and torch.cuda.is_available()):    print("C3 skipped (RUN_C3 False, no checkpoint, or no GPU)")else:    # --- CIFAR-10 -> ImageFolder (needed only by this cell) -----------------    import numpy as np, pickle    from PIL import Image    from concurrent.futures import ThreadPoolExecutor    if len(glob.glob(os.path.join(DATA, "train", "*", "*.png"))) < 100:        cp = gdrive_download(CIFAR_GDRIVE_ID, os.path.join(DOWNLOAD, "cifar10.bin"))        ce = os.path.join(DOWNLOAD, "cifar_extract")        if not glob.glob(os.path.join(ce, "**", "*batch*"), recursive=True):            unpack(cp, ce)        def _read(paths):            xs, ys = [], []            for p in paths:                with open(p, "rb") as f:                    d = pickle.load(f, encoding="bytes")                xs.append(np.asarray(d[b"data"], np.uint8).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1))                ys.append(np.asarray(d[b"labels"], np.int64))            return np.concatenate(xs), np.concatenate(ys)        g = lambda pat: sorted(p for p in glob.glob(os.path.join(ce, "**", pat), recursive=True)                               if not p.endswith(".bin"))        for split, paths in (("train", g("data_batch_*")), ("val", g("test_batch"))):            x, y = _read(paths)            dirs = [os.path.join(DATA, split, f"{k}_{CIFAR_NAMES[k]}") for k in range(10)]            for d in dirs: os.makedirs(d, exist_ok=True)            cnt, jobs = [0]*10, []            for i in range(len(x)):                k = int(y[i]); jobs.append((x[i], os.path.join(dirs[k], f"{cnt[k]:05d}.png"))); cnt[k] += 1            with ThreadPoolExecutor(max_workers=8) as ex:                list(ex.map(lambda j: Image.fromarray(j[0]).save(j[1], compress_level=1), jobs))            print(f"{split}: {len(jobs)} PNGs")    # --- patch train.py: skip FID eval, size the memory bank to 10 classes --    tp = pathlib.Path(WORK, "train.py"); s = tp.read_text(encoding="utf-8")    pat = [("        if (step % eval_per_step == 0) or (step == 1) or (step == total_steps):",            '        _skip = os.environ.get("PTFLOW_SKIP_EVAL", "0") == "1"\n'            '        if (not _skip) and ((step % eval_per_step == 0) or (step == 1) or (step == total_steps)):'),           ("    memory_bank_positive = ArrayMemoryBank(num_classes=1000, max_size=positive_bank_size)",            '    _nc = int(os.environ.get("PTFLOW_NUM_CLASSES", "1000"))\n'            '    memory_bank_positive = ArrayMemoryBank(num_classes=_nc, max_size=positive_bank_size)')]    for old, new in pat:        if new.splitlines()[0] not in s:            assert s.count(old) == 1            s = s.replace(old, new)    tp.write_text(s, encoding="utf-8")    tenv = dict(os.environ)    tenv.update(DRIFT_COMPILE="0", PTFLOW_SKIP_EVAL="1",                PTFLOW_NUM_CLASSES=str(NUM_CLASSES), PYTHONPATH=WORK,                OMP_NUM_THREADS="2", NCCL_P2P_DISABLE="1", NCCL_IB_DISABLE="1",                PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TQDM_MININTERVAL="30")    NP = min(2, torch.cuda.device_count())    target = RESUME_STEP + STEPS_C3    print(f"{len(ARMS)} arms x {STEPS_C3} steps  ~{len(ARMS)*STEPS_C3*3.73/3600:.1f} h total\n")    for name, kw in ARMS.items():        rd = ARM_DIRS[name]        os.makedirs(os.path.join(rd, "checkpoints"), exist_ok=True)        # already finished in an earlier pass?  don't pay for it twice.        mp = os.path.join(rd, "log", "metrics.jsonl")        if os.path.exists(mp):            done = [json.loads(l)["step"] for l in open(mp) if l.strip()]            if done and max(done) >= target - 20:                print(f"arm {name}: already at step {max(done)} -- skipping"); continue        dst = os.path.join(rd, "checkpoints", f"state_{RESUME_STEP:08d}.pt")        if not os.path.exists(dst):            # An arm that drops the scale net cannot load the saved pt_optimizer:            # train.py:564 builds it over potential.parameters() + scale_net            # .parameters(), and ckpt_util.py:95 loads pt_optimizer            # unconditionally (only the scale *model* load is guarded, line 91).            # So strip the theta-side optimizer state for those arms.  The            # generator's own optimizer state is untouched; only the potential's            # Adam moments restart.            if str(kw["scale_mode"]).lower() == "none":                pay = torch.load(CKPTS[RESUME_STEP], map_location="cpu", weights_only=False)                dropped = [k for k in ("pt_optimizer", "pt_scale_model",                                       "pt_ema_scale_model") if k in pay]                for k in dropped: pay.pop(k)                torch.save(pay, dst); del pay                print(f"arm {name}: seed written without {dropped}")            else:                shutil.copy2(CKPTS[RESUME_STEP], dst)        cp = os.path.join(WORK, "configs", "gen", f"_arm_{name}.yaml")        pathlib.Path(cp).write_text(yaml.safe_dump(            base_config(target, kw["lam"], kw["scale_mode"], kw["logw_clip"]), sort_keys=False))        print("=" * 78); print(f"ARM {name}:  {kw}"); print("=" * 78)        cmd = ([sys.executable, "-m", "torch.distributed.run", "--standalone",                f"--nproc_per_node={NP}"] if NP > 1 else [sys.executable]) + \              ["train.py", "--config", cp, "--workdir", rd]        t0 = time.time()        p = subprocess.Popen(cmd, cwd=WORK, env=tenv,                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)        buf = b""        while True:            ch = p.stdout.read(512)            if not ch: break            buf += ch.replace(b"\r", b"\n")            *lines, buf = buf.split(b"\n")            for ln in lines[-2:]:                print(ln.decode("utf-8", "replace"), flush=True)        p.wait()        print(f"arm {name}: exit {p.returncode} after {(time.time()-t0)/60:.1f} min\n")

### C3 — the comparison

In [ ]:
import json, os, matplotlib.pyplot as pltseries = {}for name, rd in ARM_DIRS.items():    mp = os.path.join(rd, "log", "metrics.jsonl")    if os.path.exists(mp):        series[name] = [json.loads(l) for l in open(mp) if l.strip()]if rows_m:    series["C_original (lam=0.3, clip=30)"] = rows_mif not series:    print("no arms to compare -- run C3 first")else:    panels = [("loss_0.05/global", "OT drift loss", False),              ("pt/ess", "ESS / K  (honest once clip=0)", False),              ("pt/logw_spread", "log-weight spread", True),              ("pt/prox_resid_rel", "prox_resid_rel", True),              ("pt/clip_frac", "clip_frac", False),              ("pt/health", "health (2=healthy)", False)]    fig, axes = plt.subplots(2, 3, figsize=(15, 7))    for ax, (k, title, logy) in zip(axes.ravel(), panels):        for name, rs in series.items():            v = [(r["step"], r[k]) for r in rs if k in r]            if v: ax.plot(*zip(*v), lw=1.2, label=name)        ax.set_title(title, fontsize=10); ax.set_xlabel("step")        if logy: ax.set_yscale("log")        if k == "pt/ess":            ax.axhline(0.30, color="tab:orange", ls="--", lw=1)            ax.axhline(0.05, color="tab:red", ls="--", lw=1)    axes.ravel()[0].legend(fontsize=7)    fig.suptitle("C3: two arms branched from the same step-25000 checkpoint", fontsize=12)    fig.tight_layout(); fig.savefig(os.path.join(FIGS, "C3_arm_comparison.png")); plt.show()    import statistics as st    tail = lambda rs, k: st.mean(r[k] for r in rs[-len(rs)//4:] if k in rs[-1])    save_table("C3_arm_comparison",        ["arm", "drift loss (last 25%)", "ESS", "clip_frac", "logw_spread", "prox_resid_rel"],        [[n, f"{tail(rs,'loss_0.05/global'):.4f}", f"{tail(rs,'pt/ess'):.4f}",          f"{tail(rs,'pt/clip_frac'):.4f}", f"{tail(rs,'pt/logw_spread'):.4g}",          f"{tail(rs,'pt/prox_resid_rel'):.2f}"] for n, rs in series.items()])

## [D1] Matched-seed sample gridsSame seed, same `cfg_scale`, three checkpoints. This is the only honest way toclaim the samples are improving — one grid on its own says nothing.

In [ ]:
import os, sys, torch, numpy as np, matplotlib.pyplot as pltif not CKPTS or not torch.cuda.is_available():    print("D1 skipped (no checkpoints or no GPU)")else:    if WORK not in sys.path: sys.path.insert(0, WORK)    os.chdir(WORK); os.environ["DRIFT_COMPILE"] = "0"    from inference import _load_model, make_sampler    @torch.no_grad()    def grid(ckpt, cfg_scale=1.5, per_class=8, seed=0):        model, post, step, dev, *_ = _load_model(ckpt, CFG_REF, want_potential=False)        labels = torch.arange(NUM_CLASSES, device=dev).repeat_interleave(per_class)        rng = torch.Generator(device=dev); rng.manual_seed(seed)        x = make_sampler(model, None, None, mode="A", cfg_scale=cfg_scale)(labels, rng)        img = post(x).float().cpu().numpy().transpose(0, 2, 3, 1)        del model; torch.cuda.empty_cache()        return np.clip(img, 0, 1).reshape(NUM_CLASSES, per_class, 32, 32, 3), step    shown = sorted(CKPTS)    fig, axes = plt.subplots(1, len(shown), figsize=(5.0 * len(shown), 7.0))    for ax, s in zip(np.atleast_1d(axes), shown):        g, st_ = grid(CKPTS[s])        canvas = g.transpose(0, 2, 1, 3, 4).reshape(NUM_CLASSES * 32, g.shape[1] * 32, 3)        ax.imshow(canvas); ax.set_xticks([])        ax.set_yticks(np.arange(NUM_CLASSES) * 32 + 16)        ax.set_yticklabels(CIFAR_NAMES, fontsize=9)        ax.set_title(f"step {st_}", fontsize=11)    fig.suptitle("Mode A, cfg_scale = 1.5, identical seed and noise across checkpoints",                 fontsize=12)    fig.tight_layout(); fig.savefig(os.path.join(FIGS, "D1_matched_seed_grids.png")); plt.show()    print("\nCompare columns, not rows.  If step 1000 and step 25000 look the same at a")    print("matched seed, that is the finding -- do not claim progress the figure denies.")

## [E] Package everything

In [ ]:
import os, glob, shutil, jsonmanifest = {"figures": sorted(os.path.basename(p) for p in glob.glob(f"{FIGS}/*.png")),            "tables":  sorted(os.path.basename(p) for p in glob.glob(f"{TABS}/*"))}with open(os.path.join(RESULTS, "manifest.json"), "w") as f:    json.dump(manifest, f, indent=2)zp = shutil.make_archive("/kaggle/working/ptflow_results", "zip", RESULTS)print("figures:")for f in manifest["figures"]: print("  ", f)print("tables:")for f in manifest["tables"]:  print("  ", f)print(f"\n{zp}  ({os.path.getsize(zp)/2**20:.1f} MiB)")print("\nDownload it from the notebook's Output tab.")